# **Problem Statement**

## Context

Business communities in the United States are facing high demand for human resources, but one of the constant challenges is identifying and attracting the right talent, which is perhaps the most important element in remaining competitive. Companies in the United States look for hard-working, talented, and qualified individuals both locally as well as abroad.

The Immigration and Nationality Act (INA) of the US permits foreign workers to come to the United States to work on either a temporary or permanent basis. The act also protects US workers against adverse impacts on their wages or working conditions by ensuring US employers' compliance with statutory requirements when they hire foreign workers to fill workforce shortages. The immigration programs are administered by the Office of Foreign Labor Certification (OFLC).

OFLC processes job certification applications for employers seeking to bring foreign workers into the United States and grants certifications in those cases where employers can demonstrate that there are not sufficient US workers available to perform the work at wages that meet or exceed the wage paid for the occupation in the area of intended employment.

## Objective

In FY 2016, the OFLC processed 775,979 employer applications for 1,699,957 positions for temporary and permanent labor certifications. This was a nine percent increase in the overall number of processed applications from the previous year. The process of reviewing every case is becoming a tedious task as the number of applicants is increasing every year.

The increasing number of applicants every year calls for a Machine Learning based solution that can help in shortlisting the candidates having higher chances of VISA approval. OFLC has hired the firm EasyVisa for data-driven solutions. You as a data  scientist at EasyVisa have to analyze the data provided and, with the help of a classification model:

* Facilitate the process of visa approvals.
* Recommend a suitable profile for the applicants for whom the visa should be certified or denied based on the drivers that significantly influence the case status.

## Data Description

The data contains the different attributes of employee and the employer. The detailed data dictionary is given below.

* case_id: ID of each visa application
* continent: Information of continent the employee
* education_of_employee: Information of education of the employee
* has_job_experience: Does the employee has any job experience? Y= Yes; N = No
* requires_job_training: Does the employee require any job training? Y = Yes; N = No
* no_of_employees: Number of employees in the employer's company
* yr_of_estab: Year in which the employer's company was established
* region_of_employment: Information of foreign worker's intended region of employment in the US.
* prevailing_wage:  Average wage paid to similarly employed workers in a specific occupation in the area of intended employment. The purpose of the prevailing wage is to ensure that the foreign worker is not underpaid compared to other workers offering the same or similar service in the same area of employment.
* unit_of_wage: Unit of prevailing wage. Values include Hourly, Weekly, Monthly, and Yearly.
* full_time_position: Is the position of work full-time? Y = Full Time Position; N = Part Time Position
* case_status:  Flag indicating if the Visa was certified or denied

## Note: This is a sample solution for the project. Projects will NOT be graded on the basis of how well the submission matches this sample solution. Projects will be graded on the basis of the rubric only.

# **Importing necessary libraries**

In [ ]:
# Installing the libraries with the specified version.
# !pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 xgboost==3.0.5 -q --user

**Note**: *After running the above cell, kindly restart the notebook kernel and run all cells sequentially from the start again.*

In [ ]:
# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# Library to split data
from sklearn.model_selection import train_test_split

# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)

# To oversample and undersample data
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Libraries different ensemble classifiers
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
)

from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier

# Libraries to get different metric scores
from sklearn import metrics
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# To tune different models
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings("ignore")

from VKPyKit.EDA import *
from VKPyKit.DT import *
from VKPyKit.MLM import *


# **Loading the dataset**

In [ ]:
visa = pd.read_csv("EasyVisa.csv")
df = visa.copy()

# **Overview of the Dataset**

In [ ]:
display(HTML("<h2>Info</h2>", df.info()))

__Observation__
1. 9 Columns are objects and 3 numeric

In [ ]:
display(HTML("<h2>Columns</h2>"),df.columns)

In [ ]:
display(HTML("<h2>Head</h2>"),df.head())
display(HTML("<h2>Tail</h2>"),df.tail())
display(HTML("<h2>Shape</h2>"),df.shape)

__Observation__
1. 25480 records in data
1. Unit of wage varies, may need to noramlized.
1. Case_id is just an identifier, and may need to be dropped.

# Exploratory Data Analysis (EDA)


## Checks

In [ ]:
## Null values
display(HTML("<h2>Is null values:</h2>"),df.isnull().sum())

__Observation__
1. There are no null values in any column

In [ ]:
display(HTML("<h2>Is duplicated values:</h2>"),df.duplicated().sum())


__Observation__
1. No duplicate values.

In [ ]:
display(HTML("<h2>Describe:</h2>"),df.describe().T)
display(HTML("<h2>Describe / include O:</h2>"),df.describe(include="O").T)


__Observations__
1. Negatives in number of employees, need fixing
1. Year of establishment may need to change to years in operation as just a number, not year. Ex. 2000 would become 25. 

In [ ]:
display(HTML("<h2>Case status count:</h2>"), df["case_status"].value_counts(normalize=True)*100, df["case_status"].value_counts())

__Observation__
1. Approx spread between Case status is 66-33 percent. 
1. More certified cases than denied.

In [ ]:
# let's check for missing values in the data
round(df.isnull().sum() / df.isnull().count() * 100, 2)

__Observation__
1. No missing values. 

In [ ]:
display(HTML("<h2>No of Employees</h2>"),df.loc[df["no_of_employees"] < 0]['no_of_employees'] , df.loc[df["no_of_employees"] < 0].shape)

__Observation__
1. 33 rows are negetive no of emplyees. 

In [ ]:
#Let's fix the negetive employee count, only 33 have negative value, so we will convert them to positive
df["no_of_employees"] = abs(df["no_of_employees"])
display(df.loc[df["no_of_employees"] < 0].shape)

In [ ]:
#Let's drop Case Id, it is unique anyways
df.drop(["case_id"], axis=1, inplace=True)

## Univariate Analysis

### Lay of the land

In [ ]:
# list of all categorical variables
cat_col = df.columns

# printing the number of occurrences of each unique value in each categorical column
for column in cat_col:
    print(df[column].value_counts(normalize=True)*100)
    print("-" * 50)

__Observation__
1. Continent - 66% are from Asia, followed by Europe at 14%, lowest is Oceania  at 0.7%
1. Education - 40% bachelors degree holders. 
1. Approx 58-41 - Job experience
1. 88% requiring training 
1. Northeast, West, and South take most of employment regions. 
1. unit of wage, may need to be normalized. 
1. 89% are in full time position. 

### Observations on education of employee

In [ ]:
EDA.barplot_labeled(data=df,feature="education_of_employee",percentages=True)

#### Note:
1. Masters and Bachelors make 78% of all records followed by highschool and doctorate at lowest 8.6%

### Observations on region of employment

In [ ]:
EDA.barplot_labeled(data=df,feature="region_of_employment",percentages=True)

#### Note
1. West, Northwest adn South has equally ~ 25% +
1. Island is the lowest

### Observations on job experience

In [ ]:
EDA.barplot_labeled(data=df,feature="has_job_experience",percentages=True)


#### Note
1. Job experience is 58% - 41% split

### Observations on no_of_employees

In [ ]:
EDA.histogram_boxplot(df,"no_of_employees",bins=10)

### Observations on yr_of_estab

In [ ]:
EDA.histogram_boxplot(df,"yr_of_estab",bins=10)

### Observations on full_time_position

In [ ]:
EDA.barplot_labeled(data=df,feature="full_time_position",percentages=True)

#### Note
1. Full time position has 89-10% split

### Observations on Continent

In [ ]:
EDA.barplot_labeled(data=df,feature="continent",percentages=True)

#### Note
1. Asia takes the most 66% and then followed by NA and Europe, and third group can be Adrica, SA, and Oceania.

### Observations on Training 

In [ ]:
EDA.barplot_labeled(data=df,feature="requires_job_training",percentages=True)

#### Note 
1. 88% require training 

### Observations on prevailing_wage

In [ ]:
EDA.histogram_boxplot(data=df,feature="prevailing_wage",bins=100)

### Observations on unit_of_wage

In [ ]:
EDA.barplot_labeled(data=df,feature="unit_of_wage",percentages=True)

### Observations on case status

In [ ]:
EDA.barplot_labeled(data=df,feature="case_status",percentages=True)

* 66.8% of the visas were certified.

## Bivariate Analysis

### __Pivot the columns first against case status.__

In [ ]:
pivot = EDA.pivot_table_all(data=df, target="case_status", chart_type="bar", figsize=(8, 4))

### Barplot Stacked

In [ ]:
EDA.barplot_stacked_all(data=df, target="case_status", predictors=
                        ["education_of_employee", "region_of_employment", "has_job_experience", "full_time_position", "continent", "requires_job_training"])

#  0   case_id                25480 non-null  object 
#  1   continent              25480 non-null  object 
#  2   education_of_employee  25480 non-null  object 
#  3   has_job_experience     25480 non-null  object 
#  4   requires_job_training  25480 non-null  object 
#  5   no_of_employees        25480 non-null  int64  
#  6   yr_of_estab            25480 non-null  int64  
#  7   region_of_employment   25480 non-null  object 
#  8   prevailing_wage        25480 non-null  float64
#  9   unit_of_wage           25480 non-null  object 
#  10  full_time_position     25480 non-null  object 
#  11  case_status            25480 non-null  object 

### Distribution Plots

In [ ]:
EDA.distribution_plot_for_target_all(data=df, target="case_status", predictors=df.columns, figsize=(12, 6))

### Heatmap All 

In [ ]:
cols_list = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(10, 5))
sns.heatmap(
    df[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral"
)
plt.show()



## EDA Observations
1. What is the distribution of visa case statuses (certified vs. denied)?
- Approx 66% are certified vs 33% Denied. 
2. How does the education level of employees impact visa approval rates?
- Higher education has better chances of getting certified. It is about 3 times more likely get visa certified if you have a doctorate than a highschool education. 
3. Is there a significant difference in visa approval rates between employees with and without prior job experience?
- Employees with prior experience are more likely to get certified status by about 17% more. 
4. How does the prevailing wage affect visa approval? Do higher wages lead to higher chances of approval?
- While the distributions overlap significantly, higher prevailing wages generally correlate with a higher likelihood of certification. The "Denied" cases have a slightly lower median and a more pronounced concentration of very low wage values, which may be a contributing factor to their denial (e.g., if the wage offered does not meet the actual prevailing wage for that specific occupation).
5. Do certain regions in the US have higher visa approval rates compared to others?
- Northeast & South Dominance: These two regions represent the bulk of the activity for both certified and denied cases.
- West : The West region shows a relatively high density in the "Denied" histogram compared to its density in the "Certified" histogram, suggesting a potentially higher risk of denial for applications in that region.
- Regional Concentration: Certified cases seem more geographically concentrated in the South and Northeast, while denials are more broadly spread across the West, Northeast, and South.

6. How does the number of employees in a company influence visa approval? Do larger companies have a higher approval rate?
- The relationship between the number of employees at a firm and the case status is heavily skewed by a few massive organizations.
- Extreme Skewness: The histograms show an incredibly high density of cases at the lower end of the employee count (small-to-medium businesses), with a "long tail" extending to nearly 600,000 employees.

7. Are visa approval rates different across various continents of employees? Which continent has the highest and lowest approval rates?
- Asia is the dominant source for both certified and denied visa applications.
- Dominant Group: Asia has the highest density by a significant margin for both "Denied" and "Certified" categories.
- Europe and North America show moderate density levels in both categories.
- Africa, South America, and Oceania represent a very small portion of the dataset.
- Statistical Distribution: The boxplots show that the median continent for both "Denied" and "Certified" cases is North America, with the interquartile range (IQR) extending up to Asia.

# **Data Pre-processing**

- Missing value treatment (if needed)
- Feature engineering (if needed)
- Outlier detection and treatment (if needed)
- Preparing data for modeling
- Any other preprocessing steps (if needed)

In [ ]:
df["case_status"] = df["case_status"].apply(lambda x: 1 if x == "Certified" else 0)

X = df.drop(["case_status"], axis=1)
y = df["case_status"]
X = pd.get_dummies(X, drop_first=True)

# Splitting data into training, validation and test sets
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=EDA.RANDOM_STATE, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=EDA.RANDOM_STATE, stratify=y_temp
)

In [ ]:
print(f"Training set : {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set : {X_test.shape}")
print(f"Percentage of classes in training set: {y_train.value_counts(normalize=True)}")
print(f"Percentage of classes in validation set: {y_val.value_counts(normalize=True)}")
print(f"Percentage of classes in test set: {y_test.value_counts(normalize=True)}")

### Observation
Based on the data distribution you provided, here is an analysis of your dataset split and class proportions:

**Split Proportions**: **60/20/20** split (approximately 15,288 training samples and 5,096 each for validation and testing).
The most impressive part of your split is the **consistency of the target class (`case_status`)** across all three sets:

* **Training**: ~66.79% (Class 1) / ~33.21% (Class 0)
* **Validation**: ~66.78% (Class 1) / ~33.22% (Class 0)
* **Testing**: ~66.80% (Class 1) / ~33.20% (Class 0)

This has **Stratified Sampling**, a best practice because it ensures that each set is a "miniature version" of the original population, preventing a scenario where the model is tested on a distribution it never saw during training.

**Presence of Moderate Imbalance** Dataset is **moderately imbalanced**, with a 2:1 ratio (Class 1 is twice as common as Class 0):

# **Model Building**

### Lets build a function to show the graph of results. 

In [ ]:
"""
# Build a function to show the graph of each model
"""
def showgraph(df_results, title):
    df_melted = df_results.melt(id_vars=['ModelName', 'Run'], 
                        value_vars=['Accuracy', 'Recall', 'Precision', 'F1'],
                        var_name='Metric', value_name='Score')
    
    # Set visual style
    metrics = ['Accuracy', 'Recall', 'Precision', 'F1']
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    sns.set_style("whitegrid")
    
    for i, metric in enumerate(metrics):
        # Assign the plot to a variable 'ax' to access its containers
        ax = sns.barplot(data=df_melted[df_melted['Metric'] == metric],
                        x='ModelName', y='Score', hue='Run', ax=axes[i], palette='viridis')
        
        # --- ADD VALUE LABELS HERE ---
        # Iterate through the bar containers (one for each 'Run')
        for container in ax.containers:
            ax.bar_label(container, fmt='%.3f', padding=3, fontsize=8, rotation=90)
          
        # -----------------------------
        
        axes[i].set_title(f'Model Comparison: {metric}', fontsize=8)
        axes[i].set_ylim(0, 1.2) # Increased limit slightly to fit labels on top
        axes[i].set_ylabel('Score')
        axes[i].set_xlabel('Model Name')
        axes[i].legend(title='Run', loc='lower right')
        axes[i].tick_params(axis='x', rotation=90)

   
    plt.tight_layout()
    plt.savefig(f'{title}.png')
    plt.show()

In [ ]:
# Setting up dataframe to store results
model_results = pd.DataFrame(columns=["Type","ModelName","Run", "Accuracy", "Recall", "Precision", "F1"])
models = []

### Models with Original Features and base models

In [ ]:
# Evaluating Original data models
models_original = []  

# Adding models to the list
models_original.append(("Bagging", BaggingClassifier(estimator=DecisionTreeClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced'), random_state=EDA.RANDOM_STATE)))
models_original.append(("Randomforest", RandomForestClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced')))
models_original.append(("GradientBoost", GradientBoostingClassifier(random_state=EDA.RANDOM_STATE)))
models_original.append(("Adaboost", AdaBoostClassifier(random_state=EDA.RANDOM_STATE)))
models_original.append(("DecisionTree", DecisionTreeClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced')))

# Fitting and evaluating original data models
for name, model in models_original:
    model.fit(X_train, y_train)

    results = MLM.model_performance_classification(model, X_train, y_train)
    results["Type"] = "Original" 
    results["ModelName"] = name 
    results["Run"] = "Training"
    model_results = pd.concat([model_results, results], ignore_index=True)

    results = MLM.model_performance_classification(model, X_val, y_val)
    results["Type"] = "Original" 
    results["ModelName"] = name 
    results["Run"] = "Validation"
    model_results = pd.concat([model_results, results], ignore_index=True)

    results = MLM.model_performance_classification(model, X_test, y_test)
    results["Type"] = "Original" 
    results["ModelName"] = name 
    results["Run"] = "Testing"
    model_results = pd.concat([model_results, results], ignore_index=True)

models.append(models_original)
display(model_results)



In [ ]:
showgraph(model_results[model_results["Type"] == "Original"], title="Model Comparison - Original Data")

### Observations
1. Overfitting: Models like DecisionTree and Randomforest have perfect or near-perfect training scores (1.0 or 0.99) but drop significantly on validation and testing (around 0.66–0.72). This suggests they are overfitting to the training data.
1. Generalization: GradientBoost and Adaboost show much more consistent scores across all runs (e.g., 0.75 training vs 0.74 testing), which indicates better generalization to new, unseen data.
1. Metrics Choice: While accuracy is common, for imbalanced datasets, you should prioritize the F1-score, which is the harmonic mean of precision and recall.

Model	
1. GradientBoost- Best Overall Performer- The highest Testing Accuracy (0.7419) - least amount of "gap" between training and testing
2. Adaboost	- Strong second place performer. Similar to GradientBoost- generalizes well- achieves the highest Recall on the Validation set (0.8921) - Good when positive cases is a priority.
3. Randomforest	- Overfit - performs better on testing than a simple Decision Tree (0.7229 vs 0.6626), 0.999 training accuracy may suggest too complex model.
4. Bagging - 	Seems to be Overfitting - slightly worse than Random Forest across most metrics
5. DecisionTree - 	Worst Performer - perfect training scores (1.00) - lowest testing accuracy (0.6626)

## Model Building - Oversampled Data

In [ ]:
print("Before Oversampling, counts of label 'Certified': {}".format(sum(y_train == 1)))
print("Before Oversampling, counts of label 'Denied': {} \n".format(sum(y_train == 0)))

sm = SMOTE( sampling_strategy=1, k_neighbors=5, random_state=EDA.RANDOM_STATE)  # Synthetic Minority Over Sampling Technique
X_train_over, y_train_over = sm.fit_resample(X_train, y_train)


print("After Oversampling, counts of label 'Certified': {}".format(sum(y_train_over == 1)))
print("After Oversampling, counts of label 'Denied': {} \n".format(sum(y_train_over == 0)))


print("After Oversampling, the shape of train_X: {}".format(X_train_over.shape))
print("After Oversampling, the shape of train_y: {} \n".format(y_train_over.shape))

### Observations
The primary goal of SMOTE is to eliminate the bias toward the majority class by equalizing the number of samples:

* **Before SMOTE:** Dataset had a moderate imbalance with 10,211 'Certified' (67%) and 5,077 'Denied' (33%) cases.
* **After SMOTE:** Now have exactly **10,211** observations.
* **Result:** 50/50 distribution - prevents a "frequency bias"
* **Growth in Volume:** from 15,288 samples to **20,422** samples.
* **Synthetic Logic:** Created *new, synthetic* examples by interpolating between existing minority points.
* **Feature Preservation:** The shape of train_X remaining at **21 columns** 

### Summary of the "After SMOTE" State

| Feature | Before SMOTE | After SMOTE |
| --- | --- | --- |
| **Certified Count** | 10,211 | 10,211 |
| **Denied Count** | 5,077 | 10,211 |
| **Total Rows** | 15,288 | 20,422 |
| **Class Ratio** | 2:1 (Imbalanced) | 1:1 (Balanced) |



In [ ]:
# Evaluating Oversampled data models
models_over = []  

models_over.append(("Bagging", BaggingClassifier(estimator=DecisionTreeClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced'), random_state=EDA.RANDOM_STATE)))
models_over.append(("RandomForest", RandomForestClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced')))
models_over.append(("GradientBoost", GradientBoostingClassifier(random_state=EDA.RANDOM_STATE)))
models_over.append(("Adaboost", AdaBoostClassifier(random_state=EDA.RANDOM_STATE)))
models_over.append(("DecisionTree", DecisionTreeClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced')))

# Fitting and evaluating oversampled data models
for name, model in models_over:
    model.fit(X_train_over, y_train_over)

    results = MLM.model_performance_classification(model, X_train, y_train)
    results["Type"] = "OverSampled" 
    results["ModelName"] = name 
    results["Run"] = "Training"
    model_results = pd.concat([model_results, results], ignore_index=True)

    results = MLM.model_performance_classification(model, X_val, y_val)
    results["Type"] = "OverSampled" 
    results["ModelName"] = name 
    results["Run"] = "Validation"
    model_results = pd.concat([model_results, results], ignore_index=True)

    results = MLM.model_performance_classification(model, X_test, y_test)
    results["Type"] = "OverSampled" 
    results["ModelName"] = name 
    results["Run"] = "Testing"
    model_results = pd.concat([model_results, results], ignore_index=True)

models.append(models_over)
display(model_results[model_results["Type"] == "OverSampled"])

In [ ]:
showgraph(model_results[model_results["Type"] == "OverSampled"], title="Model Comparison OverSampled")

### Observations for Oversampling
1. Analysis of Model Overfitting
- Presence of high variance (overfitting) in several models.
- Extreme Overfitting: The DecisionTree and Randomforest models show perfect or near-perfect training scores across all metrics - drop significantly on Validation and Testing data
- - DecisionTree Training Accuracy: 1.000000 vs. Testing Accuracy: 0.662677.
- - Randomforest Training Accuracy: 0.999935 vs. Testing Accuracy: 0.722920.
- - Moderate Overfitting: Bagging also shows a notable performance gap, with training scores near 0.98 dropping to approximately 0.69–0.70 during testing.

2. Generalization and Stability
- The boosting models seems to have better generalization - consistent across Training, Validation, and Testing runs
- GradientBoost Performance: most stability, with a training accuracy of 0.759288 and a testing accuracy of 0.741954.
- Adaboost Performance: consistent performance, with testing scores (Accuracy: 0.730181)

3. Key Metric Comparison
- Recall vs. Precision: particularly Adaboost and GradientBoost, exhibit higher Recall (approx. 0.87–0.89 on validation) than Precision (approx. 0.75–0.77 on validation). This suggests the models are more effective at identifying positive cases but may generate more false positives.
- F1-Score:  GradientBoost (Testing F1: 0.818945) and Adaboost (Testing F1: 0.813509) are effective overall models despite having lower training scores than the overfitted tree models.

## Model Building - Undersampled Data

In [ ]:
# Evaluating Undersampled data models
rus = RandomUnderSampler(random_state=EDA.RANDOM_STATE)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

In [ ]:
print("Before Under Sampling, counts of label 'Certified': {}".format(sum(y_train == 1)))
print("Before Under Sampling, counts of label 'Denied': {} \n".format(sum(y_train == 0)))

print("After Under Sampling, counts of label 'Certified': {}".format(sum(y_train_under == 1)))
print("After Under Sampling, counts of label 'Denied': {} \n".format(sum(y_train_under == 0)))

print("After Under Sampling, the shape of train_X: {}".format(X_train_under.shape))
print("After Under Sampling, the shape of train_y: {} \n".format(y_train_under.shape))

In [ ]:
# Evaluating Undersampled data models
models_under = []  # Empty list to store all the models
# Adding models to the list
models_under.append(("Bagging", BaggingClassifier(estimator=DecisionTreeClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced'), random_state=EDA.RANDOM_STATE)))
models_under.append(("RandomForest", RandomForestClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced')))
models_under.append(("GradientBoost", GradientBoostingClassifier(random_state=EDA.RANDOM_STATE)))
models_under.append(("Adaboost", AdaBoostClassifier(random_state=EDA.RANDOM_STATE)))
models_under.append(("DecisionTree", DecisionTreeClassifier(random_state=EDA.RANDOM_STATE, class_weight='balanced')))


for name, model in models_under:
    model.fit(X_train_under, y_train_under)

    results = MLM.model_performance_classification(model, X_train, y_train)
    results["Type"] = "UnderSampled" 
    results["ModelName"] = name 
    results["Run"] = "Training"
    model_results = pd.concat([model_results, results], ignore_index=True)

    results = MLM.model_performance_classification(model, X_val, y_val)
    results["Type"] = "UnderSampled" 
    results["ModelName"] = name 
    results["Run"] = "Validation"
    model_results = pd.concat([model_results, results], ignore_index=True)

    results = MLM.model_performance_classification(model, X_test, y_test)
    results["Type"] = "UnderSampled" 
    results["ModelName"] = name 
    results["Run"] = "Testing"
    model_results = pd.concat([model_results, results], ignore_index=True)

models.append(models_under)
model_results[model_results["Type"] == "UnderSampled"]

In [ ]:
showgraph(model_results[model_results["Type"] == "UnderSampled"], title="Model Comparison UnderSampled Data")

### Observations for Undersampling
- High Overfitting: The DecisionTree, Randomforest, and Bagging models show significant overfitting - near-perfect training scores (Accuracy $\approx$ 0.98–1.00) - sharp decline in performance on the Validation and Testing sets (Accuracy $\approx$ 0.65–0.72).
- Strong Generalization: GradientBoost and Adaboost most stable models - training scores are lower than the tree-based models - performance  consistent across all runs (Training, Validation, and Testing), looks like captured generalizable patterns rather than memorizing noise.

Metric Insights
- Best Overall Performer: GradientBoost is the best model - highest testing Accuracy, Precision, and F1-score.
- Recall vs. Precision: Most models have higher Recall than Precision (particularly Adaboost and GradientBoost) - effective at identifying positive cases but have a higher tendency for "false alarms" (false positives).
- F1-Score Utility: The F1-score confirms that despite lower training scores, the boosting models provide a better real-world balance between precision and recall than the overfitted tree models.

## Oversampling vs undersampling comparision

In [ ]:
model_results[((model_results["Type"] == "UnderSampled") | (model_results["Type"] == "OverSampled")) 
    & ((model_results["Run"] == "Testing") | (model_results["Run"] == "Validation"))].sort_values(by=["F1"], ascending=False)

#### Observations
This comprehensive analysis evaluates the performance of 5 models—**Adaboost, GradientBoost, RandomForest, Bagging, and DecisionTree**—trained using two different resampling techniques (**OverSampling** and **UnderSampling**). 

#### Overall Performance Leaders (F1-Score)
- The **F1-Score** is a reliable metric here as it balances Precision and Recall for an imbalanced dataset. 
- The results consistently show that **Boosting** algorithms (Adaboost and GradientBoost) combined with **OverSampling** are the better performers.
- **Best Overall Model:** **OverSampled Adaboost** achieved the highest Testing F1-score (**0.8128**) and the highest Testing Recall (**0.8763**).
- **Best Accuracy:** **OverSampled GradientBoost** provided the most balanced results with the highest Testing Accuracy (**0.7349**) and a very strong F1-score (**0.8108**).


#### Generalization Analysis (Validation vs. Testing)

* **Adaboost (OverSampled):** F1 dropped only **0.0065** (from 0.8193 to 0.8128).
* **GradientBoost (OverSampled):** F1 dropped only **0.0061** (from 0.8169 to 0.8108).
* **DecisionTree (UnderSampled):** Remained the least stable, showing the lowest performance across the board.

#### Summary Table of Top Testing Performers

The table below ranks the top 5 model configurations based on their Testing set F1-Scores:

| Rank | Model Name | Technique | Accuracy | Recall | Precision | F1-Score |
| --- | --- | --- | --- | --- | --- | --- |
| **1** | **Adaboost** | **OverSampled** | 0.7303 | **0.8763** | 0.7578 | **0.8128** |
| **2** | **GradientBoost** | **OverSampled** | **0.7348** | 0.8504 | 0.7746 | **0.8108** |
| **3** | **RandomForest** | **OverSampled** | 0.7142 | 0.8131 | 0.7714 | 0.7917 |
| **4** | **GradientBoost** | **UnderSampled** | 0.6979 | 0.7176 | **0.8086** | 0.7604 |
| **5** | **Adaboost** | **UnderSampled** | 0.6885 | 0.7270 | 0.7899 | 0.7572 |


#### Recommendations
1. **Use GradientBoost for Stability:** If the goal is a balanced model that generalizes well without complex resampling, the **Original GradientBoost** is the best choice.
2. **Use Adaboost with OverSampling for Recall:** If your business case prioritizes finding as many positive cases as possible (high Recall), the **OverSampled Adaboost** is highly effective.
3. **Use UnderSampling for Precision:** If "False Alarms" (False Positives) are very costly, use **UnderSampled GradientBoost**, as it offers the highest Precision.
4. **For High Sensitivity (Maximum Detection):** Deploy **OverSampled Adaboost**. It is the most effective at catching the "minority" cases, making it ideal if the cost of missing a case is high.
5. **For Balanced Reliability:** Deploy **OverSampled GradientBoost**. It offers the highest overall accuracy while maintaining a top-tier F1-score.
6. **For Precision-Critical Tasks:** If "false alarms" carry a very high cost, consider **UnderSampled GradientBoost**, as its Precision of **0.8086** is the highest recorded in testing.
7. **Avoid DecisionTrees:** Regardless of resampling, simple Decision Trees consistently provided the lowest Accuracy and F1-scores, proving inadequate compared to ensemble methods.


# **Model Performance Improvement**

### **Note** - Using the parameters provided to hypertune. 

1. Sample parameter grids have been provided to do necessary hyperparameter tuning. These sample grids are expected to provide a balance between model performance improvement and execution time. One can extend/reduce the parameter grid based on execution time and system configuration.
  - Please note that if the parameter grid is extended to improve the model performance further, the execution time will increase
2. The models chosen in this notebook are based on test runs. One can update the best models as obtained upon code execution and tune them for best performance.

- For Gradient Boosting:

    ```python
    param_grid = {
        "init": [AdaBoostClassifier(random_state=1),DecisionTreeClassifier(random_state=1)],
        "n_estimators": np.arange(50,110,25),
        "learning_rate": [0.01,0.1,0.05],
        "subsample":[0.7,0.9],
        "max_features":[0.5,0.7,1],
    }
    ```

- For Adaboost:

    ```python   
    param_grid = {
        "n_estimators": np.arange(50,110,25),
        "learning_rate": [0.01,0.1,0.05],
        "base_estimator": [
            DecisionTreeClassifier(max_depth=2, random_state=1),
            DecisionTreeClassifier(max_depth=3, random_state=1),
        ],
    }
    ```

- For Bagging Classifier:

    ```python
        param_grid = {
            'max_samples': [0.8,0.9,1],
            'max_features': [0.7,0.8,0.9],
            'n_estimators' : [30,50,70],
        }
        ```
- For Random Forest:

    ```python
    param_grid = {
        "n_estimators": [50,110,25],
        "min_samples_leaf": np.arange(1, 4),
        "max_features": [np.arange(0.3, 0.6, 0.1),'sqrt'],
        "max_samples": np.arange(0.4, 0.7, 0.1)
    }
    ```

- For Decision Trees:

    ```python
    param_grid = {
        'max_depth': np.arange(2,6),
        'min_samples_leaf': [1, 4, 7],
        'max_leaf_nodes' : [10, 15],
        'min_impurity_decrease': [0.0001,0.001]
    }
    ```

- For XGBoost:

    ```python
    param_grid={
        'n_estimators':np.arange(50,110,25),
        'scale_pos_weight':[1,2,5],
        'learning_rate':[0.01,0.1,0.05],
        'gamma':[1,3],
        'subsample':[0.7,0.9]
    }
    ```


## Hyperparameter Tuning - Random Forest

In [ ]:
%%time
# Choose the type of classifier.
randomforrest_tuned = RandomForestClassifier(random_state=EDA.RANDOM_STATE, oob_score=True, bootstrap=True)

parameters = {
    "n_estimators": [50,110,25],
    "min_samples_leaf": np.arange(1, 4),
    "max_features": [np.arange(0.3, 0.6, 0.1),'sqrt'],
    "max_samples": np.arange(0.4, 0.7, 0.1)
}

# Type of scoring used to compare parameter combinations
acc_scorer = metrics.make_scorer(metrics.f1_score)

# Run the grid search
grid_obj = GridSearchCV(randomforrest_tuned, parameters, scoring=acc_scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train_over, y_train_over)

# Set the clf to the best combination of parameters
randomforrest_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
randomforrest_tuned.fit(X_train, y_train)

results = MLM.model_performance_classification(randomforrest_tuned, X_train, y_train)
results["Type"] = "OverTuned" 
results["ModelName"] = "RandomForestTuned" 
results["Run"] = "Training"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(randomforrest_tuned, X_val, y_val)
results["Type"] = "OverTuned" 
results["ModelName"] = "RandomForestTuned"  
results["Run"] = "Validation"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(randomforrest_tuned, X_test, y_test)
results["Type"] = "OverTuned" 
results["ModelName"] = "RandomForestTuned"  
results["Run"] = "Testing"
model_results = pd.concat([model_results, results], ignore_index=True)

models.append(("RandomForestTuned", randomforrest_tuned))


MLM.plot_confusion_matrix(model=randomforrest_tuned, predictors=X_train, expected=y_train, 
                            title="Confusion Matrix for RandomForest Tuned Model on Training Set")
MLM.plot_confusion_matrix(model=randomforrest_tuned, predictors=X_val, expected=y_val, 
                            title="Confusion Matrix for RandomForest Tuned Model on Validation Set")
MLM.plot_confusion_matrix(model=randomforrest_tuned, predictors=X_test, expected=y_test, 
                            title="Confusion Matrix for RandomForest Tuned Model on Test Set")

MLM.plot_feature_importance(model=randomforrest_tuned, features=X_train.columns)




In [ ]:
display(model_results[model_results["Type"] == "OverTuned"])
showgraph(model_results[model_results["Type"] == "OverTuned"], title="Model Comparison Oversampled Tuned Data")

## Hyperparameter Tuning - AdaBoost Classifier

In [ ]:
%%time
# Choose the type of classifier.
adaboost_tuned = AdaBoostClassifier(random_state=EDA.RANDOM_STATE)

# Grid of parameters to choose from
parameters = {
    "n_estimators": np.arange(50,110,25),
    "learning_rate": [0.01,0.1,0.05],
    "estimator": [
        DecisionTreeClassifier(max_depth=2, random_state=EDA.RANDOM_STATE),
        DecisionTreeClassifier(max_depth=3, random_state=EDA.RANDOM_STATE),
    ],
}

# Type of scoring used to compare parameter combinations
acc_scorer = metrics.make_scorer(metrics.f1_score)

# Run the grid search
grid_obj = GridSearchCV(adaboost_tuned, parameters, scoring=acc_scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train_over, y_train_over)


# Set the clf to the best combination of parameters
adaboost_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
adaboost_tuned.fit(X_train, y_train)

results = MLM.model_performance_classification(adaboost_tuned, X_train, y_train)
results["Type"] = "OverTuned" 
results["ModelName"] = "AdaBoostTuned" 
results["Run"] = "Training"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(adaboost_tuned, X_val, y_val)
results["Type"] = "OverTuned" 
results["ModelName"] = "AdaBoostTuned"  
results["Run"] = "Validation"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(adaboost_tuned, X_test, y_test)
results["Type"] = "OverTuned" 
results["ModelName"] = "AdaBoostTuned"  
results["Run"] = "Testing"
model_results = pd.concat([model_results, results], ignore_index=True)

models.append(("AdaBoostTuned", adaboost_tuned))


MLM.plot_confusion_matrix(model=adaboost_tuned, predictors=X_train, expected=y_train, 
                            title="Confusion Matrix for AdaBoost Tuned Model on Training Set")
MLM.plot_confusion_matrix(model=adaboost_tuned, predictors=X_val, expected=y_val, 
                            title="Confusion Matrix for AdaBoost Tuned Model on Validation Set")
MLM.plot_confusion_matrix(model=adaboost_tuned, predictors=X_test, expected=y_test, 
                            title="Confusion Matrix for AdaBoost Tuned Model on Test Set")

MLM.plot_feature_importance(model=adaboost_tuned, features=X_train.columns)

In [ ]:
display(model_results[model_results["Type"] == "OverTuned"])
showgraph(model_results[model_results["Type"] == "OverTuned"], title="Model Comparison Oversampled Tuned Data")

## Hyperparameter Tuning - Gradient Boosting Classifier

In [ ]:
%%time
# Choose the type of classifier.
gradientboost_tuned = GradientBoostingClassifier(init=AdaBoostClassifier(random_state=EDA.RANDOM_STATE), random_state=EDA.RANDOM_STATE)
    
# Grid of parameters to choose from
parameters = {
    "init": [AdaBoostClassifier(random_state=1),DecisionTreeClassifier(random_state=1)],
    "n_estimators": np.arange(50,110,25),
    "learning_rate": [0.01,0.1,0.05],
    "subsample":[0.7,0.9],
    "max_features":[0.5,0.7,1],
}

# Type of scoring used to compare parameter combinations
acc_scorer = metrics.make_scorer(metrics.f1_score)

# Run the grid search
grid_obj = GridSearchCV(gradientboost_tuned, parameters, scoring=acc_scorer, cv=5, n_jobs=-1)
grid_obj = grid_obj.fit(X_train_over, y_train_over)


# Set the clf to the best combination of parameters
gradientboost_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
gradientboost_tuned.fit(X_train, y_train)

results = MLM.model_performance_classification(gradientboost_tuned, X_train, y_train)
results["Type"] = "OverTuned" 
results["ModelName"] = "GradientBoostTuned" 
results["Run"] = "Training"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(gradientboost_tuned, X_val, y_val)
results["Type"] = "OverTuned" 
results["ModelName"] = "GradientBoostTuned"  
results["Run"] = "Validation"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(gradientboost_tuned, X_test, y_test)
results["Type"] = "OverTuned" 
results["ModelName"] = "GradientBoostTuned"  
results["Run"] = "Testing"
model_results = pd.concat([model_results, results], ignore_index=True)

models.append(("GradientBoostTuned", gradientboost_tuned))

MLM.plot_confusion_matrix(model=gradientboost_tuned, predictors=X_train, expected=y_train, 
                            title="Confusion Matrix for Gradient Boost Tuned Model on Training Set")
MLM.plot_confusion_matrix(model=gradientboost_tuned, predictors=X_val, expected=y_val, 
                            title="Confusion Matrix for Gradient Boost Tuned Model on Validation Set")
MLM.plot_confusion_matrix(model=gradientboost_tuned, predictors=X_test, expected=y_test, 
                            title="Confusion Matrix for Gradient Boost Tuned Model on Test Set")

MLM.plot_feature_importance(model=gradientboost_tuned, features=X_train.columns)

In [ ]:
display(model_results[model_results["Type"] == "OverTuned"])
showgraph(model_results[model_results["Type"] == "OverTuned"].sort_values(by="Type", ascending=False), title="Model Comparison Oversampled Tuned Data")

## Hyperparameter Tuning - XGBoost Classifier

In [ ]:
%%time
# Choose the type of classifier.
xgb_tuned = XGBClassifier(random_state=EDA.RANDOM_STATE, eval_metric="logloss")

# Grid of parameters to choose from
parameters = {
    'n_estimators':np.arange(50,110,25),
    'scale_pos_weight':[1,2,5],
    'learning_rate':[0.01,0.1,0.05],
    'gamma':[1,3],
    'subsample':[0.7,0.9]
}

# Type of scoring used to compare parameter combinations
acc_scorer = metrics.make_scorer(metrics.f1_score)

# Run the grid search
grid_obj = GridSearchCV(estimator=xgb_tuned,param_grid=parameters,scoring=acc_scorer,cv=5,n_jobs=5)  ## Complete the code to define the grid search object
grid_obj = grid_obj.fit(X_train_over, y_train_over)

# Set the clf to the best combination of parameters
xgb_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
xgb_tuned.fit(X_train, y_train)


# Set the clf to the best combination of parameters
xgb_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
xgb_tuned.fit(X_train, y_train)

results = MLM.model_performance_classification(xgb_tuned, X_train, y_train)
results["Type"] = "OverTuned" 
results["ModelName"] = "XGradientBoostTuned" 
results["Run"] = "Training"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(xgb_tuned, X_val, y_val)
results["Type"] = "OverTuned" 
results["ModelName"] = "XGradientBoostTuned"  
results["Run"] = "Validation"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(xgb_tuned, X_test, y_test)
results["Type"] = "OverTuned" 
results["ModelName"] = "XGradientBoostTuned"  
results["Run"] = "Testing"
model_results = pd.concat([model_results, results], ignore_index=True)

models.append(("XGradientBoostTuned", xgb_tuned))

MLM.plot_confusion_matrix(model=xgb_tuned, predictors=X_train, expected=y_train, 
                            title="Confusion Matrix for X Gradient Boost Tuned Model on Training Set")
MLM.plot_confusion_matrix(model=xgb_tuned, predictors=X_val, expected=y_val, 
                            title="Confusion Matrix for X Gradient Boost Tuned Model on Validation Set")
MLM.plot_confusion_matrix(model=xgb_tuned, predictors=X_test, expected=y_test, 
                            title="Confusion Matrix for X Gradient Boost Tuned Model on Test Set")

MLM.plot_feature_importance(model=xgb_tuned, features=X_train.columns)

In [ ]:
display(model_results[model_results["Type"] == "OverTuned"])
showgraph(model_results[model_results["Type"] == "OverTuned"].sort_values(by="Type", ascending=False), title="Model Comparison Oversampled Tuned Data")

## Hyperparameter Tuning - Bagging Classifier

In [ ]:
%%time
# Choose the type of classifier.
bagging_tuned = BaggingClassifier(random_state=EDA.RANDOM_STATE)

# Grid of parameters to choose from
parameters = {
    'max_samples': [0.8,0.9,1],
    'max_features': [0.7,0.8,0.9],
    'n_estimators' : [30,50,70],
}

# Type of scoring used to compare parameter combinations
acc_scorer = metrics.make_scorer(metrics.f1_score)

# Run the grid search
grid_obj = GridSearchCV(estimator=bagging_tuned,param_grid=parameters,scoring=acc_scorer,cv=5,n_jobs=5)  ## Complete the code to define the grid search object
grid_obj = grid_obj.fit(X_train_over, y_train_over)

# Set the clf to the best combination of parameters
bagging_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
bagging_tuned.fit(X_train, y_train)


# Set the clf to the best combination of parameters
bagging_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
bagging_tuned.fit(X_train, y_train)

results = MLM.model_performance_classification(bagging_tuned, X_train, y_train)
results["Type"] = "OverTuned" 
results["ModelName"] = "BaggingTuned" 
results["Run"] = "Training"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(bagging_tuned, X_val, y_val)
results["Type"] = "OverTuned" 
results["ModelName"] = "BaggingTuned"  
results["Run"] = "Validation"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(bagging_tuned, X_test, y_test)
results["Type"] = "OverTuned" 
results["ModelName"] = "BaggingTuned"  
results["Run"] = "Testing"
model_results = pd.concat([model_results, results], ignore_index=True)

models.append(("BaggingTuned", bagging_tuned))

MLM.plot_confusion_matrix(model=bagging_tuned, predictors=X_train, expected=y_train, 
                            title="Confusion Matrix for Bagging Tuned Model on Training Set")
MLM.plot_confusion_matrix(model=bagging_tuned, predictors=X_val, expected=y_val, 
                            title="Confusion Matrix for Bagging Tuned Model on Validation Set")
MLM.plot_confusion_matrix(model=bagging_tuned, predictors=X_test, expected=y_test, 
                            title="Confusion Matrix for Bagging Boost Tuned Model on Test Set")

In [ ]:
display(model_results[model_results["Type"] == "OverTuned"])
showgraph(model_results[model_results["Type"] == "OverTuned"].sort_values(by="Type", ascending=False), title="Model Comparison Oversampled Tuend Data")

## Hyperparameter Tuning - Decision Tree

In [ ]:
%%time
# Choose the type of classifier.
decisiontree_tuned = DecisionTreeClassifier(random_state=EDA.RANDOM_STATE)

# Grid of parameters to choose from
parameters = {
    'max_depth': np.arange(2,6),
    'min_samples_leaf': [1, 4, 7],
    'max_leaf_nodes' : [10, 15],
    'min_impurity_decrease': [0.0001,0.001]
}

# Type of scoring used to compare parameter combinations
acc_scorer = metrics.make_scorer(metrics.f1_score)

# Run the grid search
grid_obj = GridSearchCV(estimator=decisiontree_tuned,param_grid=parameters,scoring=acc_scorer,cv=5,n_jobs=5)  ## Complete the code to define the grid search object
grid_obj = grid_obj.fit(X_train_over, y_train_over)

# Set the clf to the best combination of parameters
decisiontree_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
decisiontree_tuned.fit(X_train, y_train)


# Set the clf to the best combination of parameters
decisiontree_tuned = grid_obj.best_estimator_

# Fit the best algorithm to the data.
decisiontree_tuned.fit(X_train, y_train)

results = MLM.model_performance_classification(decisiontree_tuned, X_train, y_train)
results["Type"] = "OverTuned" 
results["ModelName"] = "DecisionTreeTuned" 
results["Run"] = "Training"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(decisiontree_tuned, X_val, y_val)
results["Type"] = "OverTuned" 
results["ModelName"] = "DecisionTreeTuned"  
results["Run"] = "Validation"
model_results = pd.concat([model_results, results], ignore_index=True)

results = MLM.model_performance_classification(decisiontree_tuned, X_test, y_test)
results["Type"] = "OverTuned" 
results["ModelName"] = "DecisionTreeTuned"  
results["Run"] = "Testing"
model_results = pd.concat([model_results, results], ignore_index=True)

models.append(("DecisionTreeTuned", decisiontree_tuned))


MLM.plot_confusion_matrix(model=decisiontree_tuned, predictors=X_train, expected=y_train, 
                            title="Confusion Matrix for DecisionTree Tuned Model on Training Set")
MLM.plot_confusion_matrix(model=decisiontree_tuned, predictors=X_val, expected=y_val, 
                            title="Confusion Matrix for DecisionTree Tuned Model on Validation Set")
MLM.plot_confusion_matrix(model=decisiontree_tuned, predictors=X_test, expected=y_test, 
                            title="Confusion Matrix for DecisionTree Boost Tuned Model on Test Set")

MLM.plot_feature_importance(model=decisiontree_tuned, features=X_train.columns)

In [ ]:
display(model_results[model_results["Type"] == "OverTuned"])
showgraph(model_results[model_results["Type"] == "OverTuned"].sort_values(by="Type", ascending=False), title="Model Comparison Oversampled Tuned Data")

# **Model Comparison and Final Model Selection**

In [ ]:
# Type	ModelName	Run	Accuracy	Recall	Precision	F1
#df_T = model_results[(model_results["Type"] == "OverTuned") & ((model_results["Run"] == "Testing")| (model_results["Run"] == "Validation"))].sort_values(by="F1", ascending=False)
df_T = model_results[(model_results["Type"] == "OverTuned")].sort_values(by="F1", ascending=False)
display(df_T)

showgraph(df_T, title="Model Comparison Oversampled Tuned Data")
# model_results

# **Actionable Insights and Recommendations**

#### Model Performance Comparative Analysis
The following analysis evaluates how each tuned model generalized across different datasets.
| Model Name | Run Type | Accuracy | Recall | Precision | F1-Score |
| --- | --- | --- | --- | --- | --- |
| **XGradientBoost (Tuned)** | Training | 0.755298 | 0.974048 | 0.741022 | 0.841704 |
|  | Validation | 0.724097 | 0.961505 | 0.719595 | 0.823145 |
|  | **Testing** | **0.719388** | **0.952115** | **0.718944** | **0.819262** |
| **AdaBoost (Tuned)** | Training | 0.751897 | 0.873959 | 0.780752 | 0.824731 |
|  | Validation | 0.744702 | 0.874816 | 0.772845 | 0.820675 |
|  | **Testing** | **0.741758** | **0.872503** | **0.771028** | **0.818633** |
| **DecisionTree (Tuned)** | Training | 0.734563 | 0.908138 | 0.748245 | 0.820474 |
|  | Validation | 0.726452 | 0.905084 | 0.741990 | 0.815462 |
|  | **Testing** | **0.727041** | **0.907168** | **0.741773** | **0.816175** |
| **GradientBoost (Tuned)** | Training | 0.757457 | 0.877387 | 0.784845 | 0.828540 |
|  | Validation | 0.746860 | 0.877755 | 0.773634 | 0.822412 |
|  | **Testing** | **0.738030** | **0.870153** | **0.768353** | **0.816090** |
| **RandomForest (Tuned)** | Training | 0.859956 | 0.945843 | 0.858794 | 0.900219 |
|  | Validation | 0.742739 | 0.868645 | 0.773822 | 0.818496 |
|  | **Testing** | **0.731162** | **0.860752** | **0.765813** | **0.810512** |
| **Bagging (Tuned)** | Training | 0.998888 | 0.999902 | 0.998435 | 0.999168 |
|  | Validation | 0.725471 | 0.893329 | 0.745829 | 0.812943 |
|  | **Testing** | **0.719388** | **0.889835** | **0.741675** | **0.809028** |


#### 1. Overfitting and Generalization
* **Extreme Overfitting:** **Bagging (Tuned)** and **RandomForest (Tuned)** exhibit significant overfitting. Bagging achieves a near-perfect Training F1 of 0.999 - drops to 0.809 on Testing. RandomForest drops from a 0.900 Training F1 to 0.810 on Testing.
* **Best Generalization:** **XGradientBoost**, **AdaBoost**, and **GradientBoost** show excellent stability, with very narrow gaps between training and testing metrics, indicating robust performance on unseen data.

#### 2. Trade-offs
* **High Sensitivity (Recall):** **XGradientBoost (Tuned)** is the strongest model for identifying the majority class, maintaining a testing Recall of **0.952**.
* **High Precision:** **AdaBoost (Tuned)** and **GradientBoost (Tuned)** offer the highest precision on testing (0.771 and 0.768, respectively), making them suitable if the cost of "False Positives" is high.

#### 3. Best Model : XGradientBoost (Tuned)
The **XGradientBoost (Tuned)** model is the recommended choice for deployment.
* **Top Performance:** It achieved the highest overall **Testing F1-score (0.819)**.
* **Exceptional Recall:** Its ability to correctly identify 95.2% of positive cases makes it highly effective for real-world scenarios where missing a case is expensive.
* **Robustness:** The model shows minimal decay from validation to testing, proving its reliability.
#### 4. Strategic Recommendations
* **Prioritize Boosting:** The boosting models (XGB, AdaBoost, GradientBoost) outperforms the bagging models in generalization.
* **Address Precision Needs:** If the case requires fewer false alarms, we should use **AdaBoost (Tuned)**, it maintains the highest Accuracy (0.741) and Precision (0.771) with a reasonable F1-score.
* **Regularize Bagging Models:** To use Bagging or RandomForest effectively, further pruning or stricter constraints on tree depth are needed to curb the drastic drop between training and testing.




In [ ]:
!jupyter nbconvert --to html "Project_Full_Code_Notebook_EasyVisa.ipynb"